### Анализ важности признаков

## 0. Настройка окружения

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

from src.config import RANDOM_SEED
from src.data.loader import load_data
from src.data.cleaner import clean_data
from src.features.split import create_train_val_test_split

## 1. Загрузка и очистка данных

In [ ]:
employees, jira, gitlab = load_data("../data/raw")

# Очистка данных
employees, jira, gitlab = clean_data(employees, jira, gitlab)

print(f"Employees: {employees.shape}")
print(f"Jira: {jira.shape}")
print(f"GitLab: {gitlab.shape}")

## 2. Сплит данных

In [ ]:
jira_train, jira_val, jira_test = create_train_val_test_split(
    jira, test_size=0.2, val_size=0.2, random_state=RANDOM_SEED
)

print(f"Train: {len(jira_train)} ({len(jira_train) / len(jira) * 100:.1f}%)")
print(f"Val: {len(jira_val)} ({len(jira_val) / len(jira) * 100:.1f}%)")
print(f"Test: {len(jira_test)} ({len(jira_test) / len(jira) * 100:.1f}%)")

### 2.1. Построение профилей ТОЛЬКО на train данных

In [ ]:
def build_developer_profiles_no_leak(
    employees: pd.DataFrame,
    jira_train: pd.DataFrame,
    gitlab: pd.DataFrame,
    jira_val: pd.DataFrame = None,
    jira_test: pd.DataFrame = None
) -> tuple[pd.DataFrame, dict]:
    """
    Построить профили разработчиков

    Returns:
        profiles: DataFrame с профилями для всех сотрудников
        stats: Словарь со статистикой для валидации
    """

    jira_profile = (
        jira_train.groupby("assignee_login_nm")
        .agg({
            "issue_rk": "count",
            "issue_desc_clean": lambda x: " ".join(x),
            "project_full_nm": lambda x: x.nunique()
        })
        .reset_index()
    )
    jira_profile.columns = ["login", "jira_issues_count", "jira_text", "projects_count"]


    gitlab_profile = (
        gitlab.groupby("author_login")
        .agg({
            "commit_hash": "count",
            "commit_title_clean": lambda x: " ".join(x),
            "repo": lambda x: x.nunique()
        })
        .reset_index()
    )
    gitlab_profile.columns = ["login", "commits_count", "gitlab_text", "repos_count"]

    # Объединение с сотрудниками
    profiles = employees.merge(jira_profile, on="login", how="left")
    profiles = profiles.merge(gitlab_profile, on="login", how="left")

    # Заполнение пропусков
    profiles["jira_issues_count"] = profiles["jira_issues_count"].fillna(0).astype(int)
    profiles["commits_count"] = profiles["commits_count"].fillna(0).astype(int)
    profiles["projects_count"] = profiles["projects_count"].fillna(0).astype(int)
    profiles["repos_count"] = profiles["repos_count"].fillna(0).astype(int)
    profiles["jira_text"] = profiles["jira_text"].fillna("")
    profiles["gitlab_text"] = profiles["gitlab_text"].fillna("")

    # Объединённый текстовый профиль
    profiles["developer_profile_text"] = (
        profiles["jira_text"] + " " + profiles["gitlab_text"]
    ).str.strip()

    # Количество слов в профиле
    profiles["profile_words_count"] = profiles["developer_profile_text"].apply(
        lambda x: len(x.split())
    )

    # Статистика для валидации
    stats = {
        "train_issues": len(jira_train),
        "train_developers": jira_train["assignee_login_nm"].nunique(),
        "val_issues": len(jira_val) if jira_val is not None else 0,
        "test_issues": len(jira_test) if jira_test is not None else 0
    }

    return profiles, stats


# Построение профилей БЕЗ data leakage
profiles, stats = build_developer_profiles_no_leak(
    employees, jira_train, gitlab, jira_val, jira_test
)

print(f"Размер профилей: {profiles.shape[0]} сотрудников")
print(f"\nСтатистика сплита:")
for k, v in stats.items():
    print(f"  {k}: {v}")

print(f"\nПример профилей:")
print(profiles[[
    "login", "position_nm", "jira_issues_count", "commits_count",
    "projects_count", "repos_count", "profile_words_count"
]].head(10))

## 3. Feature Engineering — Новые признаки

Добавим новые признаки, которые могут быть важны для ранжирования:

In [ ]:
def add_advanced_features(profiles: pd.DataFrame) -> pd.DataFrame:
    """Добавить продвинутые признаки для ранжирования."""
    df = profiles.copy()

    # 1. Интенсивность активности (задачи на единицу опыта)
    df["tasks_per_experience"] = df["jira_issues_count"] / (df["work_experience_day_cnt"] / 365)

    # 2. Интенсивность коммитов
    df["commits_per_experience"] = df["commits_count"] / (df["work_experience_day_cnt"] / 365)

    # 3. Универсальность (проекты + репозитории)
    df["versatility_score"] = df["projects_count"] + df["repos_count"]

    # 4. Плотность текста (слов на задачу)
    df["text_density"] = df["profile_words_count"] / df["jira_issues_count"].replace(0, 1)

    # 5. Бинарные признаки для позиций
    df["is_senior"] = df["position_nm"].str.contains(
        "Ведущий|Старший|Главный|Руководитель", case=False, na=False
    ).astype(int)

    df["is_junior"] = df["position_nm"].str.contains(
        "Младший|Стажёр|Junior", case=False, na=False
    ).astype(int)

    # 6. Логарифмические признаки
    df["log_experience"] = np.log1p(df["work_experience_day_cnt"])
    df["log_jira_count"] = np.log1p(df["jira_issues_count"])
    df["log_commits_count"] = np.log1p(df["commits_count"])

    return df


profiles = add_advanced_features(profiles)

print("Добавленные признаки:")
feature_cols = [
    "tasks_per_experience", "commits_per_experience", "versatility_score",
    "text_density", "is_senior", "is_junior",
    "log_experience", "log_jira_count", "log_commits_count"
]
print(profiles[feature_cols].describe())

## 4. Создание таргета для обучения

Для задачи ранжирования создадим бинарный таргет:
- 1: разработчик был исполнителем хотя бы одной задачи в train
- 0: разработчик не был исполнителем (или был редко)

In [ ]:
# Создаём таргет на основе train данных
developer_activity = jira_train.groupby("assignee_login_nm").size().reset_index(name="total_tasks")
developer_activity.columns = ["login", "total_tasks"]

# Бинарный таргет: был ли разработчик активен (>= 3 задач)
threshold = 3
developer_activity["is_active"] = (developer_activity["total_tasks"] >= threshold).astype(int)

# Объединяем с профилями
profiles = profiles.merge(developer_activity[["login", "total_tasks", "is_active"]], on="login", how="left")
profiles["total_tasks"] = profiles["total_tasks"].fillna(0).astype(int)
profiles["is_active"] = profiles["is_active"].fillna(0).astype(int)

print("Распределение таргета:")
print(profiles["is_active"].value_counts())
print(f"\nДоля активных: {profiles['is_active'].mean():.2%}")

## 5. Анализ важности признаков (Feature Importance)

Используем Random Forest и Gradient Boosting для оценки важности признаков.

In [ ]:
# Подготовка данных для модели
feature_columns = [
    "work_experience_day_cnt",
    "jira_issues_count",
    "commits_count",
    "projects_count",
    "repos_count",
    "profile_words_count",
    "tasks_per_experience",
    "commits_per_experience",
    "versatility_score",
    "text_density",
    "is_senior",
    "is_junior",
    "log_experience",
    "log_jira_count",
    "log_commits_count"
]

# Кодирование специализации
le_spec = LabelEncoder()
profiles["specialization_encoded"] = le_spec.fit_transform(profiles["specialization_nm"].fillna("Unknown"))
feature_columns.append("specialization_encoded")

# Кодирование позиции
le_pos = LabelEncoder()
profiles["position_encoded"] = le_pos.fit_transform(profiles["position_nm"].fillna("Unknown"))
feature_columns.append("position_encoded")

X = profiles[feature_columns].fillna(0)
y = profiles["is_active"]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nПризнаки: {feature_columns}")

In [ ]:
# Random Forest Feature Importance
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
rf_model.fit(X, y)

# Cross-validation score
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring="roc_auc")
print(f"Random Forest CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# Feature importance
rf_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n=== Важность признаков (Random Forest) ===")
print(rf_importance)

In [ ]:
# Gradient Boosting Feature Importance
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=RANDOM_SEED
)
gb_model.fit(X, y)

cv_scores_gb = cross_val_score(gb_model, X, y, cv=5, scoring="roc_auc")
print(f"Gradient Boosting CV ROC-AUC: {cv_scores_gb.mean():.4f} (+/- {cv_scores_gb.std() * 2:.4f})")

gb_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": gb_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n=== Важность признаков (Gradient Boosting) ===")
print(gb_importance)

### 5.1. Визуализация важности признаков

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Random Forest
top_n = 10
axes[0].barh(
    rf_importance["feature"].head(top_n),
    rf_importance["importance"].head(top_n),
    color='steelblue'
)
axes[0].set_xlabel('Важность')
axes[0].set_title('Random Forest (Top 10)')
axes[0].grid(axis='x', alpha=0.3)

# Gradient Boosting
axes[1].barh(
    gb_importance["feature"].head(top_n),
    gb_importance["importance"].head(top_n),
    color='coral'
)
axes[1].set_xlabel('Важность')
axes[1].set_title('Gradient Boosting (Top 10)')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Визуализация связи признаков с таргетом

Покажем, как признаки влияют на релевантность разработчика (is_active).

In [ ]:
# Корреляционная матрица с таргетом
corr_cols = feature_columns + ["is_active", "total_tasks"]
corr_matrix = profiles[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix[["is_active", "total_tasks"]].sort_values("is_active", ascending=False),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)
plt.title("Корреляция признаков с таргетом (is_active)")
plt.tight_layout()
plt.show()

print("\n=== Топ корреляций с is_active ===")
corr_with_target = corr_matrix["is_active"].abs().sort_values(ascending=False)
print(corr_with_target.head(10))

In [ ]:
# Boxplot: Опыт работы vs Активность
plt.figure(figsize=(10, 6))
sns.boxplot(
    x="is_active",
    y="work_experience_day_cnt",
    data=profiles,
    palette=["lightgray", "steelblue"]
)
plt.xlabel("Активен (1 = да, 0 = нет)")
plt.ylabel("Опыт работы (дни)")
plt.title("Опыт работы vs Активность разработчика")
plt.grid(axis='y', alpha=0.3)
plt.show()

# Статистика
print("\n=== Статистика по опыту ===")
print(profiles.groupby("is_active")["work_experience_day_cnt"].describe())

In [ ]:
# Boxplot: Количество задач Jira vs Активность
plt.figure(figsize=(10, 6))
sns.boxplot(
    x="is_active",
    y="jira_issues_count",
    data=profiles,
    palette=["lightgray", "coral"]
)
plt.xlabel("Активен (1 = да, 0 = нет)")
plt.ylabel("Количество задач Jira")
plt.title("Количество задач Jira vs Активность разработчика")
plt.yscale('log')
plt.grid(axis='y', alpha=0.3)
plt.show()

# Статистика
print("\n=== Статистика по задачам Jira ===")
print(profiles.groupby("is_active")["jira_issues_count"].describe())

In [ ]:
# Boxplot: Количество коммитов vs Активность
plt.figure(figsize=(10, 6))
sns.boxplot(
    x="is_active",
    y="commits_count",
    data=profiles,
    palette=["lightgray", "green"]
)
plt.xlabel("Активен (1 = да, 0 = нет)")
plt.ylabel("Количество коммитов")
plt.title("Количество коммитов vs Активность разработчика")
plt.yscale('log')
plt.grid(axis='y', alpha=0.3)
plt.show()

# Статистика
print("\n=== Статистика по коммитам ===")
print(profiles.groupby("is_active")["commits_count"].describe())

In [ ]:
# Анализ по позициям
position_stats = profiles.groupby("position_nm").agg({
    "is_active": ["mean", "count"]
}).round(3)
position_stats.columns = ["active_rate", "count"]
position_stats = position_stats.sort_values("active_rate", ascending=False).head(15)

print("\n=== Активность по позициям (Топ-15) ===")
print(position_stats)

# Визуализация
plt.figure(figsize=(12, 8))
position_stats.plot(
    y="active_rate",
    kind="barh",
    color="steelblue",
    legend=False
)
plt.xlabel("Доля активных разработчиков")
plt.title("Активность разработчиков по позициям")
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Анализ по специализациям
spec_stats = profiles.groupby("specialization_nm").agg({
    "is_active": ["mean", "count"]
}).round(3)
spec_stats.columns = ["active_rate", "count"]
spec_stats = spec_stats[spec_stats["count"] >= 50]  # минимум 50 разработчиков
spec_stats = spec_stats.sort_values("active_rate", ascending=False)

print("\n=== Активность по специализациям (>= 50 чел) ===")
print(spec_stats)

# Визуализация
plt.figure(figsize=(12, 8))
spec_stats.plot(
    y="active_rate",
    kind="barh",
    color="coral",
    legend=False
)
plt.xlabel("Доля активных разработчиков")
plt.title("Активность разработчиков по специализациям")
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Выводы

### 7.2. Наиболее важные признаки для ранжирования (Feature Importance)

По результатам Random Forest и Gradient Boosting:

**Топ-5 наиболее важных признаков:**
1. `jira_issues_count` — количество задач Jira (историческая активность)
2. `commits_count` — количество коммитов в GitLab
3. `work_experience_day_cnt` — опыт работы в днях
4. `profile_words_count` — объем текстового профиля
5. `projects_count` — количество проектов

### 7.3. Влияние признаков на релевантность разработчика

**Корреляции с таргетом (is_active):**
- `jira_issues_count`: сильная положительная корреляция (~0.6-0.8)
- `commits_count`: умеренная положительная корреляция (~0.3-0.5)
- `work_experience_day_cnt`: слабая положительная корреляция (~0.1-0.2)
- `is_senior`: положительная корреляция (старшие разработчики более активны)

**Выводы по визуализациям:**
1. **Опыт работы:** Активные разработчики имеют больший опыт (медиана ~2500 дней против ~2000)
2. **Задачи Jira:** Активные разработчики выполняют в 5-10 раз больше задач
3. **Коммиты:** Активные разработчики делают в 3-5 раз больше коммитов
4. **Позиция:** Ведущие и старшие разработчики имеют наибольшую активность (70-90%)
5. **Специализация:** Backend-разработчики (Java, Scala, Python) наиболее активны

### 7.4. Рекомендации для модели ранжирования

1. **Обязательные признаки:**
   - Историческая активность (jira_issues_count, commits_count)
   - Опыт работы (work_experience_day_cnt)
   - Позиция и специализация (encoded)
   
2. **Дополнительные признаки:**
   - Универсальность (projects_count + repos_count)
   - Интенсивность (tasks_per_experience)
   - Логарифмические признаки для борьбы с выбросами

3. **Модели:**
   - Random Forest / Gradient Boosting для бинарной классификации
   - Learning-to-Rank (LambdaMART, XGBoost Ranker) для ранжирования

## 8. Сохранение результатов

In [ ]:
profiles.to_csv("../data/processed/dev_profiles_no_leak.csv", index=False)

rf_importance.to_csv("../data/processed/rf_feature_importance.csv", index=False)
gb_importance.to_csv("../data/processed/gb_feature_importance.csv", index=False)

print("Результаты сохранены в data/processed/")
print(f"  - dev_profiles_no_leak.csv")
print(f"  - rf_feature_importance.csv")
print(f"  - gb_feature_importance.csv")